# 2.1 — Data Cleaning & Smoothing

Tahap preprocessing bagian 1: ringkasan kualitas data, penanganan missing value, dan smoothing variabel time-series (PCHIP).

**Input:** `1_data_gathering/output/1_raw_panel_data.csv`

**Output:** `2_data_preprocessing/output/2.1_cleaned_smoothed_data.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '1_data_gathering' / 'output' / '1_raw_panel_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find 1_raw_panel_data.csv in current or parent directories.')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '1_data_gathering' / 'output' / '1_raw_panel_data.csv'
output_dir = ROOT / '2_data_preprocessing' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / '2.1_cleaned_smoothed_data.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Raw shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
display(df.head())

In [ ]:
# Filter periode analisis
# (menjaga konsistensi dengan horizon modeling)
df_filtered = df[df['tahun'] > 2021].copy()

# Imputasi khusus untuk TPT (x4_tpt_pct): isi gap pendek per provinsi
# Forward-fill hingga 5 bulan, lalu satu langkah backward-fill
df_filtered['x4_tpt_pct'] = df_filtered.groupby('provinsi_id')['x4_tpt_pct'].ffill(limit=5)
df_filtered['x4_tpt_pct'] = df_filtered.groupby('provinsi_id')['x4_tpt_pct'].bfill(limit=1)

missing_x4 = df_filtered['x4_tpt_pct'].isna().sum()
print(f'Filtered shape (tahun > 2021): {df_filtered.shape[0]:,} rows × {df_filtered.shape[1]} cols')
print(f'Missing x4_tpt_pct after fill: {missing_x4:,}')

In [ ]:
# Ringkas missing values pada beberapa kolom penting
cols_to_check = ['x6_tabungan_miliar', 'x7_jumlah_kc_bank', 'x8_ldr_pct', 'x9_npl_ratio', 'x10_rasio_umkm']

for col in cols_to_check:
    missing_data = df_filtered[df_filtered[col].isna()]
    if missing_data.empty:
        print(f'No missing values found for column: {col}')
        continue

    print(f'\n--- Missing values for column: {col} ---')
    missing_summary = (
        missing_data.groupby(['provinsi_id', 'nama_provinsi'])['tanggal']
        .agg(['min', 'max', 'count'])
        .reset_index()
        .rename(columns={'min': 'start_tanggal_missing', 'max': 'end_tanggal_missing', 'count': 'num_missing_entries'})
    )
    display(missing_summary)

In [ ]:
# Smoothing variabel time-series dengan PCHIP (mengurangi pola stepwise)
sticky_cols = [
    'x3_pdrb_per_kapita',
    'x4_tpt_pct',
    'x5_penetrasi_internet_pct',
    'x6_tabungan_miliar',
    'x7_jumlah_kc_bank',
    'x8_ldr_pct',
    'x9_npl_ratio',
    'x10_rasio_umkm',
]

df_before = df_filtered.copy()

def apply_true_spline(group: pd.DataFrame) -> pd.DataFrame:
    province_id = group.name
    group = group.sort_values(['tahun', 'bulan']).copy()
    group['provinsi_id'] = province_id

    for col in sticky_cols:
        # Skip columns that are fully missing in a province.
        if group[col].isna().all():
            continue

        # Anchor points are values where the series changes.
        is_anchor = group[col] != group[col].shift(1)
        if len(is_anchor) > 0:
            is_anchor.iloc[0] = True
            is_anchor.iloc[-1] = True

        series_with_nans = group[col].where(is_anchor, np.nan)
        group[col] = series_with_nans.interpolate(method='pchip')
        group[col] = group[col].ffill().bfill()

    return group

# Apply smoothing by province and keep the result flattened for later steps.
df_spline = (
    df_before.groupby('provinsi_id', group_keys=False)
    .apply(apply_true_spline)
    .reset_index(drop=True)
)

# Kembalikan urutan waktu untuk tahap berikutnya
df_spline = df_spline.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Smoothed shape: {df_spline.shape[0]:,} rows × {df_spline.shape[1]} cols')
print(f'Unique provinces after smoothing: {df_spline["provinsi_id"].nunique():,}')

# Persist artifact untuk tahap berikutnya
assert (df_spline['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present (expected removed upstream).'

df_spline.to_csv(output_path, index=False)
print(f'Saved: {output_path}')